#### 1. Depedencies and libraries

In [1]:
!pip install -q bertopic sentence-transformers transformers torch pandas numpy matplotlib seaborn scikit-learn scipy tqdm Sastrawi

preventing kernel crash

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
import torch
torch.set_num_threads(1)

In [4]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

In [ ]:
warnings.filterwarnings("ignore") # hides warning messages
sns.set_theme(style="whitegrid", context="notebook") # sets the theme for seaborn plots
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


Declaring variables for config

In [5]:
csv_file = "psn_news_scraped.csv"
random_state = 42

embedding_model = "paraphrase-multilingual-MiniLM-L12-v2" # Indonesian + English
sentiment_model_id = "w11wo/indonesian-roberta-base-sentiment-classifier" # Indonesian text
sentiment_model_en = "cardiffnlp/twitter-roberta-base-sentiment" # English text

min_chars = 300 # dropping articles that's shorter than this number

#### 2. Data Loading

here loading data will be performed and to check the scrapped article or news data 

In [6]:
raw_data = pd.read_csv(csv_file)
print("Shape:", raw_data.shape)
raw_data.info()
raw_data.head(3)

Shape: (918, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   link          918 non-null    object
 1   title         918 non-null    object
 2   snippet       918 non-null    object
 3   date          918 non-null    object
 4   source        918 non-null    object
 5   article_url   824 non-null    object
 6   article_text  764 non-null    object
dtypes: object(7)
memory usage: 50.3+ KB


,link,title,snippet,date,source,article_url,article_text
0,https://news.google.com/rss/articles/CBMia0FVX...,Berita Kini - Menkomdigi Meutya Hafid: Satelit...,Berita Kini - Menkomdigi Meutya Hafid: Satelit...,"Tue, 12 May 2026 07:00:00 GMT",Kementerian Komunikasi dan Digital,https://portal.komdigi.go.id/kanal-publik/beri...,Menteri Komunikasi dan Digital Meutya Hafid me...
1,https://news.google.com/rss/articles/CBMikwFBV...,KSP Dukung Pemerataan Internet di Daerah 3T Me...,KSP Dukung Pemerataan Internet di Daerah 3T Me...,"Tue, 12 May 2026 07:00:00 GMT",Kantor Staf Presiden,https://www.ksp.go.id/ksp-dukung-pemerataan-in...,"Kepala Staf Kepresidenan, Jenderal TNI (Purn.)..."
2,https://news.google.com/rss/articles/CBMiswFBV...,"Perebutan 'Kue' Internet Satelit RI Memanas, P...","Perebutan 'Kue' Internet Satelit RI Memanas, P...","Wed, 13 May 2026 07:00:00 GMT",detikInet,https://inet.detik.com/telecommunication/d-848...,Agus Tri Haryanto - detikInet\nDirektur Utama ...


In [8]:
raw_data.isnull().sum()

link              0
title             0
snippet           0
date              0
source            0
article_url      94
article_text    154
dtype: int64

#### 3. Data Cleaning

from data loading, it shows that there are articles left that is not talking about psn and how many nulls are there on url and text. so this section mainly adresses that along side other cleaning methods.

Parse dates and drop rows without text. for example converts date text into datetime so later, plot trends can be made. also removing rows where artricle text is missing

In [10]:
df = raw_data.copy()

df["date"] = pd.to_datetime(df["date"], format="%a, %d %b %Y %H:%M:%S GMT", errors="coerce")
print("Rows with invalid date format:", df["date"].isnull().sum())

n0 = len(df)
df = df.dropna(subset=["article_text"]).copy()
print(f"Dropped {n0 - len(df)} rows with missing article_text")
print(f"{len(df)} rows left after dropping {n0 - len(df)} rows")

Rows with invalid date format: 0
Dropped 154 rows with missing article_text
764 rows left after dropping 154 rows


news or article data scraped a bunch of text from article / news content. the code cell below will explore if the article contains noise, eg ads, share buttons, copyright footers, “Baca Juga” links.

In [13]:
from collections import Counter

counter = Counter()
for text in df["article_text"]:
    lines = set(line.strip() for line in str(text).split("\n") if line.strip())
    counter.update(lines)

noise = pd.DataFrame(counter.most_common(30), columns=["line", "n_articles"])
noise = noise[noise["n_articles"] >= 3]
noise["line"] = noise["line"].str[:100]
noise

,line,n_articles
0,SCROLL TO CONTINUE WITH CONTENT,169
1,Editor:,85
2,Penulis:,75
3,Bagikan,75
4,Salin Pranala,75
5,Agus Tri Haryanto - detikInet,66
6,Baca Juga,60
7,"Dilarang keras mengambil konten, melakukan cra...",38
8,© 2020,37
9,ADVERTISEMENT,35
